# Dispersive water waves

*waves · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/dispersive_waves)

A wave packet on deep water, where long waves outrun short ones: $\omega = \sqrt{g k}$. Every Fourier mode just turns its phase, so the packet at any time is one FFT, a multiplication and one inverse FFT, done for all times at once. The crests run through the envelope at twice its speed.

**Vector form**

$$\eta(x, t) = \mathrm{Re}\int \hat a(k)\,e^{i(kx - \omega(k)\,t)}\,dk,\qquad \omega = \sqrt{g\,|k|},\qquad v_g = \frac{d\omega}{dk} = \tfrac12\,\frac{\omega}{k}$$

**Index (tensor) form**

$$\hat a_{nk} = \hat a_k\,e^{-i\omega_k t_n},\qquad a_{nj} = F^{-1}_{jk}\,\hat a_{nk},\qquad \eta_{nj} = \mathrm{Re}\,a_{nj}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $a_j(0)$ | `wave_packet(x, x0, sigma, k0)` | (J,) | complex packet: Gaussian envelope times $e^{ik_0x}$ |
| $\hat a_k$ | `torch.fft.fft(a0)` | (K,) | Fourier amplitudes of the starting packet |
| $\omega_k$ | `omega(k, g)` | (K,) | deep-water dispersion relation for every mode |
| $\hat a_{nk}$ | `a_hat[None, :] * exp(-1j * omega[None, :] * t[:, None])` | (N, K) | every mode at every time, by broadcasting (exact, no time stepping) |
| $F^{-1}_{jk}\hat a_{nk}$ | `torch.fft.ifft(a_hat_t, dim=-1)` | (N, J) | repeated $k$ ⇒ summed: back to real space for all times |

Linear deep-water gravity waves (depth much larger than the wavelength, no surface tension, small slope). Since $\omega \propto \sqrt k$, the group velocity $d\omega/dk$ is exactly half the phase velocity $\omega/k$: watch the red crest overtake the green envelope peak, fade out at the front and get replaced by a new crest born at the back (the red dot jumps back by whole wavelengths). Longer waves travel faster, so the packet slowly spreads and its front gets longer waves than its back (a chirp). The FFT makes the box periodic. Because each mode only rotates its phase, $\sum|\hat a_k|^2$ (the wave energy, by Parseval) is exactly conserved.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/dispersive_waves/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    g, sigma = tensors.G, params["width"]
    k0 = 2 * math.pi / params["wavelength"]
    x, k, dx = tensors.grid(2048, dev)
    a0 = tensors.wave_packet(x, tensors.X0, sigma, k0)

    times, a = solve.solve(a0, k, g, params["t_max"])
    xbar = solve.centroid(x, a)

    v_p, v_g = equation.phase_velocity(k0, g), equation.group_velocity(k0, g)
    print(
        f"phase velocity: theory {v_p:.3f} m/s, measured at the packet centre {solve.crest_speed(x, a, times):.3f} m/s"
    )
    print(f"group velocity: theory {v_g:.3f} m/s, envelope centre moved at {solve.envelope_speed(times, xbar):.3f} m/s")
    E = equation.wave_energy(a)
    print(f"relative change of sum |a|^2: {((E - E[0]).abs().max() / E[0]).item():.1e}")
    return plot.animate(times, x, a, xbar, k0, v_p, v_g, tensors.X0, sigma, out_dir / "dispersive_waves.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/dispersive_waves/equation.py
r"""Physics → tensor translation.

Linear waves on deep water obey the dispersion relation ω(k) = √(g |k|). Each Fourier mode of the surface
evolves on its own, it only turns its phase:

    \hat a_{nk} = \hat a_k \, e^{-i \omega_k t_n}

and the surface at time t_n is the inverse Fourier transform (k summed):

    a_{nj} = F^{-1}_{jk} \hat a_{nk},        \eta_{nj} = \mathrm{Re}\, a_{nj}

Phase velocity v_p = ω/k, group velocity v_g = dω/dk = ½ √(g/k) = ½ v_p.
"""

import math

import torch


def omega(k: torch.Tensor, g: float) -> torch.Tensor:
    """ω_k = √(g |k|).   (K,) → (K,)"""
    return torch.sqrt(g * k.abs())


def evolve(a0: torch.Tensor, k: torch.Tensor, t: torch.Tensor, g: float) -> torch.Tensor:
    r"""a_{nj} = F^{-1}_{jk} [\hat a_k e^{−iω_k t_n}] for every time at once.   a0: (J,), t: (N,) → (N, J) complex"""
    a_hat = torch.fft.fft(a0)  # (K,)
    phase = torch.exp(-1j * omega(k, g)[None, :] * t[:, None])  # (N, K)
    return torch.fft.ifft(a_hat[None, :] * phase, dim=-1)  # (N, J)


def phase_velocity(k0: float, g: float) -> float:
    """v_p = ω/k = √(g/k)."""
    return math.sqrt(g / k0)


def group_velocity(k0: float, g: float) -> float:
    """v_g = dω/dk = ½ √(g/k)."""
    return 0.5 * math.sqrt(g / k0)


def wave_energy(a: torch.Tensor) -> torch.Tensor:
    r"""E_n ∝ Σ_j |a_{nj}|² (= Σ_k |\hat a_{nk}|² / K by Parseval).   (N, J) → (N,)"""
    return torch.einsum("nj,nj->n", a.real, a.real) + torch.einsum("nj,nj->n", a.imag, a.imag)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/dispersive_waves/tensors.py
"""Build the concrete tensors: the periodic grid, its wavenumbers and the starting wave packet."""

import math

import torch

G = 9.81  # m/s²
BOX = 400.0  # length of the periodic box (m)
X0 = 60.0  # starting centre of the packet (m)


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, float]:
    """x (J,) on [0, BOX), the matching FFT wavenumbers k (K = J,) in rad/m, and Δx."""
    dx = BOX / n
    x = torch.arange(n, device=device, dtype=torch.float64) * dx
    k = 2 * math.pi * torch.fft.fftfreq(n, d=dx).to(device=device, dtype=torch.float64)
    return x, k, dx


def wave_packet(x: torch.Tensor, x0: float, sigma: float, k0: float) -> torch.Tensor:
    """a_j(0) = exp(−(x − x0)² / 2σ²) · e^{i k0 x}: (J,) complex. Its real part is the water surface."""
    envelope = torch.exp(-((x - x0) ** 2) / (2 * sigma**2))
    return envelope * torch.exp(1j * k0 * x)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/dispersive_waves/solve.py
"""Evolve the packet exactly in k-space and measure how fast the envelope and the crests move."""

import torch

from .equation import evolve


def solve(a0: torch.Tensor, k: torch.Tensor, g: float, t_max: float, n_times: int = 181) -> tuple:
    """Return (times (N,), a (N, J) complex) — no time stepping, every time is computed directly."""
    times = torch.linspace(0.0, t_max, n_times, device=a0.device, dtype=torch.float64)
    return times, evolve(a0, k, times, g)


def centroid(x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
    """Envelope centre x̄_n = Σ_j x_j |a_nj|² / Σ_j |a_nj|².   (N, J) → (N,)"""
    w = a.abs() ** 2
    return (w @ x) / w.sum(dim=-1)


def envelope_speed(times: torch.Tensor, xbar: torch.Tensor) -> float:
    """Least-squares slope of x̄(t): the measured group velocity."""
    t = times - times.mean()
    return (torch.sum(t * (xbar - xbar.mean())) / torch.sum(t * t)).item()


def crest_speed(x: torch.Tensor, a: torch.Tensor, times: torch.Tensor, n: int = 0) -> float:
    """Measured phase velocity at the packet centre: local frequency / local wavenumber.

    ω_loc = −Δ(arg a)/Δt between two snapshots, k_loc = Δ(arg a)/Δx between two neighbouring points.
    """
    j = int(torch.argmax(a[n].abs()))
    dt, dx = (times[n + 1] - times[n]).item(), (x[1] - x[0]).item()
    w_loc = -torch.angle(a[n + 1, j] * a[n, j].conj()).item() / dt
    k_loc = torch.angle(a[n, j + 1] * a[n, j].conj()).item() / dx
    return w_loc / k_loc


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/dispersive_waves/plot.py
"""Animate the water surface with its envelope; a red dot rides a crest (v_p), a green dot the envelope centre (v_g)."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif


def animate(times, x, a, xbar, k0: float, v_p: float, v_g: float, x0: float, sigma: float, path: Path) -> Path:
    t, xs = times.cpu().numpy(), x.cpu().numpy()
    eta, env = a.real.cpu().numpy(), a.abs().cpu().numpy()  # (N, J)
    centre = xbar.cpu().numpy()  # (N,)
    wavelength = 2 * math.pi / k0
    crest0 = wavelength * round(x0 / wavelength)  # a crest of cos(k0 x) near the packet centre at t = 0
    window = wavelength * math.ceil(1.5 * sigma / wavelength)  # keep the crest rider within ±window of the centre

    fig, ax = plt.subplots(figsize=(7, 3.8))
    line = ax.plot(xs, eta[0], color="#2563eb", lw=1.4)[0]
    env_hi = ax.plot(xs, env[0], "--", color="#9ca3af", lw=1)[0]
    env_lo = ax.plot(xs, -env[0], "--", color="#9ca3af", lw=1)[0]
    crest = ax.plot([], [], "o", color="#dc2626", ms=7, label=f"a crest, v_p = {v_p:.2f} m/s")[0]
    group = ax.plot([], [], "o", color="#16a34a", ms=7, label=f"envelope centre, v_g = {v_g:.2f} m/s")[0]
    x_hi = min(xs[-1], centre.max() + 4 * sigma + 20)
    ax.set_xlim(max(0.0, x0 - 4 * sigma - 20), x_hi)
    ax.set_ylim(-1.15, 1.45)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("surface η (relative)")
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    title = ax.set_title("deep-water wave packet   t = 0.0 s")

    def update(step: int):
        line.set_ydata(eta[step])
        env_hi.set_ydata(env[step])
        env_lo.set_ydata(-env[step])
        xc = crest0 + v_p * t[step]  # this crest moves at the phase velocity …
        xc = centre[step] + (xc - centre[step] + window) % (2 * window) - window  # … but jumps back whole wavelengths
        crest.set_data([xc], [np.interp(xc, xs, eta[step])])
        group.set_data([centre[step]], [np.interp(centre[step], xs, env[step])])
        title.set_text(f"deep-water wave packet   t = {t[step]:.1f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'wavelength': 10.0,
    'width': 10.0,
    't_max': 60.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.dispersive_waves")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.dispersive_waves.main")
problem_dir = Path("problems/waves/dispersive_waves")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
